# Declarative Solver Rule Manifests

**Status:** Approved design, pending written-spec review  
**Date:** 2026-08-20  
**Design epic:** `bd-2bc`  
**Formal profile:** `relational_lia` v1 — implemented with Z3 `qf_lia_bool_int_enum`

## Decision

Built-in solver rule catalog data will move from Rust constructors to strict, versioned YAML manifests. YAML becomes the source of truth for family/profile/rule identity, availability, strength, authorities, required facts, subject cardinality, per-rule parameter contracts, LLM/solver guidance, and executable examples.

Rust remains the source of truth for family fact/scene models, normalization, bounded unknown projection, graph preprocessing, native constraint lowering, solver resource caps, diagnostics, and persistence behavior.

This is a hybrid manifest/native-compiler architecture. It does not introduce arbitrary runtime rule loading, raw SMT in manifests, or the deferred general `expr_v1` language.

## Goals

- Make adding or documenting a built-in rule a manifest-first change.
- Eliminate duplicated family/profile membership and executable rule-ID enums.
- Generate family and top-level MCP schemas from one validated registry.
- Make availability explicit so catalog-only rules cannot accidentally become executable.
- Turn catalog examples into the conformance fixtures for every implemented hard rule.
- Preserve stable IDs and the existing `solve_rule_spec` and `solve_rules` APIs.

## Non-goals

- Loading user-authored manifests dynamically at runtime.
- Allowing YAML to name arbitrary Rust paths or inject SMT.
- Replacing typed Rust family facts, preprocessing, or constraint compilation.
- Supporting loops, graph traversal, witness construction, or map iteration in a YAML expression language.
- Implementing the optional `expr_v1` execution kind in this change.
- Changing solver status semantics, timeout attribution, or shared resource limits.

## Architecture

Each family owns one `family.yaml` plus one YAML file per rule under `rules/`. A strict `CatalogManifestV1` loader parses those files, rejects unknown fields, validates schema versions, and constructs the existing deterministic `RuleRegistry`. Profile rule membership is derived from each rule's `profile` field rather than repeated as a list.

A build step discovers tracked manifests, validates syntax and static invariants, and generates an embedded manifest index. Runtime initialization deserializes the embedded canonical data and applies the same registry cross-reference checks. No filesystem discovery occurs in the running solver.

Implemented hard rules identify a stable native handler key from a closed enum, not an arbitrary Rust path. Each family exposes a handler table keyed by that enum. Validation enforces a bijection between executable manifests and handlers. Catalog-only rules have no handler and are excluded from executable MCP rule-ID enums.

The validated registry generates:

1. family/profile/rule responses for `solve_rule_spec`;
2. per-family executable rule-ID enums;
3. per-rule subject and parameter JSON Schema;
4. the top-level `solve_rules` rule-ID enum, directly from registry availability rather than scraping nested schemas;
5. conformance cases from manifest valid/invalid examples.

Family compilers continue to parse typed requests and lower native rules into `ConstraintExpr`. Existing solver caps, timeout budgeting, attribution, projections, and diagnostics remain unchanged.

In [ ]:
flowchart TD
    SPEC["`@spec SOLVER-RULE-MANIFEST-ROUTING
@type Status = enum[executable, catalog_only, reject]
@input manifest_valid: Bool
@input implemented_hard: Bool
@input handler_present: Bool
@output status: Status`"]

    EXECUTABLE["`@branch EXECUTABLE
@when manifest_valid and implemented_hard and handler_present
@ensures EXEC_STATUS: status = executable`"]

    CATALOG_ONLY["`@branch CATALOG_ONLY
@when manifest_valid and not implemented_hard and not handler_present
@ensures CATALOG_STATUS: status = catalog_only`"]

    REJECT["`@branch REJECT
@when not manifest_valid or implemented_hard != handler_present
@ensures REJECT_STATUS: status = reject`"]

    CHECK["`@verify ROUTING_DETERMINISTIC: prove determinism
@verify ROUTING_COVERAGE: prove partition_coverage
@verify ROUTING_EXCLUSIVE: prove partition_exclusive
@verify ROUTING_STATUSES: witness each status`"]

    SPEC --> EXECUTABLE --> CHECK
    SPEC --> CATALOG_ONLY --> CHECK
    SPEC --> REJECT --> CHECK

## Manifest and build contract

### Source layout

```text
crates/spur-solver/
├── build.rs
└── src/rules/
    ├── manifest_format.rs
    └── families/
        ├── accessibility.rs
        └── accessibility/
            ├── family.yaml
            ├── rules/
            │   ├── focus_not_obscured.yaml
            │   ├── reflow.yaml
            │   ├── target_size.yaml
            │   └── text_contrast.yaml
            └── compile.rs
```

The same layout is repeated for design, policy, and resource.

### Versioned DTO

`manifest_format.rs` is self-contained and compiled both by the library and by `build.rs`. Its top-level documents use `schema_version: 1` and strict Serde deserialization with unknown-field rejection. The DTO includes closed enums for availability, strength, parameter kinds, subject cardinality, and `NativeHandlerV1`. Parameter kinds include a `native_object` variant keyed by a closed native-validator enum for structured values such as accessibility exceptions; manifests cannot name Rust paths.

A family manifest contains family ID/version/summary and profile ID/version/summary records. A rule manifest contains rule ID/version, family, profile, primitive, summary, availability/reason, strength, authorities, required facts, LLM guidance, solver guidance, subject contract, parameter contracts, optional native handler, public valid/invalid catalog examples, and executable conformance vectors.

### Build pipeline

1. `build.rs` discovers sorted `family.yaml` and `rules/*.yaml` files beneath the four built-in family directories.
2. It parses each file with `serde_yml`, validates schema versions and manifest-only invariants, and emits `cargo:rerun-if-changed` directives.
3. It rejects duplicate IDs, missing family/profile owners, invalid profile ownership, duplicate handler keys, implemented-hard rules without handlers, catalog-only rules with handlers, invalid parameter defaults/bounds, and missing executable conformance vectors for implemented-hard rules.
4. It writes one canonical JSON bundle to `OUT_DIR`. YAML parsing is therefore a build dependency only.
5. Runtime uses `include_str!` on that generated bundle, deserializes with existing `serde_json`, converts to catalog types, and passes through `RuleRegistry::new`.

### Native execution

`NativeHandlerV1` is a closed, versioned enum. Rust dispatch is exhaustive over this enum and calls focused family-native functions. Manifests cannot name Rust paths. The build validator compares used handlers with `NativeHandlerV1::ALL`; every handler is used exactly once.

The formal routing gate above is normative:

- valid implemented-hard manifest + handler → executable;
- valid non-executable manifest + no handler → catalog-only;
- malformed manifest or handler/availability mismatch → build rejection.

## Schema and validation behavior

The public MCP tool remains a Bedrock-compatible simple object. It must not gain a top-level or per-binding `oneOf`. Its family and executable rule-ID enums come directly from the validated registry, filtered by availability and strength; it no longer scrapes IDs from family compiler schemas.

Manifest parameter contracts provide:

- type: integer, boolean, string, string enum, string array, or native object backed by a closed validator enum;
- required/optional;
- default;
- inclusive integer minimum/maximum;
- allowed enum values;
- array length limits.

A shared pre-dispatch validator checks subject cardinality, accepted parameter names, required parameters, bounds, defaults, and executable availability before invoking the family-native handler. Native-object fields delegate structural validation to the selected closed Rust validator. Family compilers retain semantic checks that depend on facts, cross-fields, typed exceptions, graph closure, selected map keys, or solver-variable construction.

Catalog guidance formulas remain explanatory strings. They are never parsed as executable constraints.

## Error handling

- YAML syntax, schema-version, ownership, availability/handler, and static contract errors fail the build with file and rule context.
- Embedded bundle or registry conversion failure remains a deterministic built-in initialization failure.
- Request contract errors become stable family compile diagnostics and never reach Z3.
- Native semantic errors preserve existing family diagnostic wording where tests assert it.
- Unknown, timeout, error, and ended solver statuses remain inconclusive; no manifest layer rewrites solver status.

## Migration

1. Add the shared manifest format, build pipeline, and equivalence tests while keeping Rust catalog constructors temporarily.
2. Extract accessibility and design manifests, switch their registries/schema IDs/examples to manifests, and prove serialized catalog equivalence.
3. Extract policy and resource manifests, including the catalog-only `rbac.minimum_privilege` rule.
4. Route top-level rule IDs from the registry and remove schema scraping.
5. Introduce exhaustive native-handler dispatch and shared static parameter/subject validation; retain native rule bodies.
6. Remove obsolete Rust rule constructors and duplicated example factories after equivalence and conformance tests pass.

## Testing and acceptance

- Unit tests reject malformed manifests for every invariant above.
- Golden equivalence tests compare all existing family/profile/rule catalog JSON before and after extraction.
- Every implemented hard manifest contributes one valid and one invalid executable request in its `conformance` vectors; the existing family-neutral conformance harness consumes those vectors, while public `examples` preserve the exact serialized `solve_rule_spec` catalog output.
- The catalog-only policy rule is present in `solve_rule_spec` and absent from executable rule enums.
- Public tool schemas remain Bedrock compatible.
- Existing platform catalog, conformance, execution, audited regression, MCP, persistence, real-Z3, formatting, and clippy checks pass.
- Stable family/profile/rule IDs and serialized guidance content do not change.

## Risks and mitigations

- **Generated artifact drift:** canonical JSON is always rebuilt from sorted YAML inputs; rerun directives cover every manifest directory.
- **YAML ambiguity:** use a strict supported subset, quote stable IDs and URLs, reject unknown fields and duplicate semantic IDs, and canonicalize before embedding.
- **Diagnostic regressions:** retain native semantic validators and add exact-error regression tests around the shared contract validator.
- **Build-script/library divergence:** the self-contained versioned DTO and static validator are shared by path between both compilation contexts.
- **Large coupled migration:** land infrastructure first, then families in dependency order with catalog-equivalence gates after each phase.
- **Accidental DSL expansion:** executable formulas remain native Rust; `expr_v1` requires a separate approved design.

## Proof evidence

| Formal unit | Cell | Profile | Obligations | Result |
|---|---|---|---:|---|
| `SOLVER-RULE-MANIFEST-ROUTING` | `67c213e6-c0e4-4e8f-a13d-391a4e9e978c` | `relational_lia` v1 | 6/6 matched | verified |

Fresh proof hashes:

- source: `3fa487a1c63fea9c61940d05a9993254756c44821311c8b48fb01c9215920059`
- IR: `15d9338dd97b70172cffa4b47038b759f377b596ee12882af4f147e7758b09e3`
- obligation set: `b2502ba305a34df81b2eb06632d1eaa230c5554693c4cb797332bb0083fd15b7`
- report: `c6be3a89682fe3bb9b471ed7e4506468f691596ae640e8064f75fd78d749b8d5`

The solver proved routing coverage, exclusivity, and determinism by unsatisfiable counterexample searches, and produced witnesses for executable, catalog-only, and rejected states.